# Strings

In [1]:
import polars as pl

In [2]:
df = pl.DataFrame(
    {
        "language": ["English", "Dutch", "Portuguese", "Finish"],
        "fruit": ["pear", "peer", "pêra", "päärynä"],
    }
)

result = df.with_columns(
    pl.col("fruit").str.len_bytes().alias("byte_count"),
    pl.col("fruit").str.len_chars().alias("letter_count"),
)

result

language,fruit,byte_count,letter_count
str,str,u32,u32
"""English""","""pear""",4,4
"""Dutch""","""peer""",4,4
"""Portuguese""","""pêra""",5,4
"""Finish""","""päärynä""",10,7


## Parsing strings

In [4]:
result = df.select(
    pl.col("fruit"),
    pl.col("fruit").str.starts_with("p").alias("starts_with_p"),
    pl.col("fruit").str.contains("p..r").alias("p..r"),
    pl.col("fruit").str.contains("e+").alias("e+"),
    pl.col("fruit").str.ends_with("r").alias("ends_with_r"),
)

result

fruit,starts_with_p,p..r,e+,ends_with_r
str,bool,bool,bool,bool
"""pear""",true,true,true,true
"""peer""",true,true,true,true
"""pêra""",true,false,false,false
"""päärynä""",true,true,false,false


## Pattern extraction

In [5]:
df = pl.DataFrame(
    {
        "urls": [
            "http://vote.com/ballon_dor?candidate=messi&ref=polars",
            "http://vote.com/ballon_dor?candidat=jorginho&ref=polars",
            "http://vote.com/ballon_dor?candidate=ronaldo&ref=polars",
        ]
    }
)

result = df.select(
    pl.col("urls").str.extract(r"candidate=(\w+)", group_index=1),
)

result


urls
str
"""messi"""
null
"""ronaldo"""


In [6]:
df = pl.DataFrame({"text": ["123 bla 45 asd", "xyz 678 910t"]})

result = df.select(
    pl.col("text").str.extract_all(r"(\d+)").alias("extracted_nrs"),
)

result

extracted_nrs
list[str]
"[""123"", ""45""]"
"[""678"", ""910""]"


## Replacement

In [7]:
df = pl.DataFrame({"text": ["123abc", "abc456"]})

result = df.with_columns(
    pl.col("text").str.replace(r"\d", "-"),
    pl.col("text").str.replace_all(r"\d", "-").alias("text_replace_all"),
)

result

text,text_replace_all
str,str
"""-23abc""","""---abc"""
"""abc-56""","""abc---"""


## Modifying strings

In [8]:
addresses = pl.DataFrame(
    {
        "addresses": [
            "128 PERF st",
            "Rust blVD, 158",
            "PoLaRs Av, 12",
            "1042 Query sq",
        ]
    }
)

addresses = addresses.select(
    pl.col("addresses").alias("originals"),
    pl.col("addresses").str.to_titlecase(),
    pl.col("addresses").str.to_lowercase().alias("lower"),
    pl.col("addresses").str.to_uppercase().alias("upper"),
)


addresses

originals,addresses,lower,upper
str,str,str,str
"""128 PERF st""","""128 Perf St""","""128 perf st""","""128 PERF ST"""
"""Rust blVD, 158""","""Rust Blvd, 158""","""rust blvd, 158""","""RUST BLVD, 158"""
"""PoLaRs Av, 12""","""Polars Av, 12""","""polars av, 12""","""POLARS AV, 12"""
"""1042 Query sq""","""1042 Query Sq""","""1042 query sq""","""1042 QUERY SQ"""


## Stripping characters

In [11]:
addr = pl.col("addresses")
chars = ", 0123456789"

result = addresses.select(
    addr.str.strip_chars(chars).alias("strip"),
    addr.str.strip_chars_end(chars).alias("end"),
    addr.str.strip_chars_start(chars).alias("start"),
    addr.str.strip_prefix("128 ").alias("prefix"),
    addr.str.strip_suffix(", 158").alias("suffix"),
)

result

strip,end,start,prefix,suffix
str,str,str,str,str
"""Perf St""","""128 Perf St""","""Perf St""","""Perf St""","""128 Perf St"""
"""Rust Blvd""","""Rust Blvd""","""Rust Blvd, 158""","""Rust Blvd, 158""","""Rust Blvd"""
"""Polars Av""","""Polars Av""","""Polars Av, 12""","""Polars Av, 12""","""Polars Av, 12"""
"""Query Sq""","""1042 Query Sq""","""Query Sq""","""1042 Query Sq""","""1042 Query Sq"""


## Slicing

In [12]:
df = pl.DataFrame(
    {
        "fruits": ["pear", "mango", "dragonfruit", "passionfruit"],
        "n": [1, -1, 4, -4],
    }
)

result = df.with_columns(
    pl.col("fruits").str.slice(pl.col("n")).alias("slice"),
    pl.col("fruits").str.head(pl.col("n")).alias("head"),
    pl.col("fruits").str.tail(pl.col("n")).alias("tail"),
)

result

fruits,n,slice,head,tail
str,i64,str,str,str
"""pear""",1,"""ear""","""p""","""r"""
"""mango""",-1,"""o""","""mang""","""ango"""
"""dragonfruit""",4,"""onfruit""","""drag""","""ruit"""
"""passionfruit""",-4,"""ruit""","""passionf""","""ionfruit"""
